# 01 — N-gram Language Model

**Objective:** implement unigram, bigram, and trigram language models from scratch; use add-one smoothing; generate text; and evaluate negative log-likelihood and perplexity.

This notebook uses only the Python standard library. Put your own corpus in a UTF-8 text file named `corpus.txt` beside this notebook. If that file is absent, a tiny built-in demo corpus is used so the notebook can run end-to-end. The demo corpus is only for checking the implementation, not for drawing meaningful quality conclusions.

## Workflow
1. Load text and split it into train, validation, and test portions.
2. Tokenize and build the vocabulary from training text only.
3. Estimate n-gram counts.
4. Use Laplace smoothing to avoid zero probabilities.
5. Generate text and evaluate held-out perplexity.

## 1. Imports and configuration

We keep this implementation intentionally small and readable. The tokenizer lowercases text and keeps words and punctuation as separate tokens.

In [1]:
import math
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

SEED = 42
random.seed(SEED)

CORPUS_PATH = Path("/content/corpus.txt")

## 2. Load the corpus

For a meaningful experiment, use a reasonably sized plain-text corpus. For example, a public-domain book or a collection of domain-specific documents.

The fallback corpus below is deliberately small and is only a smoke test. Replace it with your own `corpus.txt` before interpreting perplexity or comparing model quality.

In [2]:
DEMO_TEXT = """
A language model predicts the next token from the tokens that came before it.
A statistical language model estimates probabilities from observed text.
A bigram model uses one previous token to predict the next token.
A trigram model uses two previous tokens to predict the next token.
A unigram model predicts tokens without using preceding context.
Training data is used to estimate the model parameters.
Validation data helps compare settings without using the final test set.
Test data should be kept separate until final evaluation.
Smoothing assigns some probability to events that were not observed in training.
Add one smoothing adds one to each n gram count.
A vocabulary maps tokens to the words and symbols represented by the model.
Unknown tokens are mapped to a special unknown token.
Perplexity summarizes the average uncertainty assigned to a sequence.
Lower perplexity means higher average probability on the evaluated tokens.
Text generation repeatedly samples or selects a token from a probability distribution.
The generated token is appended to the context before predicting again.
A larger n gram can use more context but may have sparse counts.
A small corpus is not enough to estimate a useful language model.
Careful data splitting prevents evaluation data from leaking into training.
Autoregressive models factor a sequence into conditional next token probabilities.
"""

if CORPUS_PATH.exists():
    raw_text = CORPUS_PATH.read_text(encoding="utf-8")
    corpus_source = str(CORPUS_PATH)
else:
    raw_text = DEMO_TEXT.strip()
    corpus_source = "built-in demo corpus (replace with corpus.txt for meaningful experiments)"

print("Corpus source:", corpus_source)
print("Characters:", len(raw_text))
print(raw_text[:500])

Corpus source: /content/corpus.txt
Characters: 10831
# Naruto Character Corpus

**Executive Summary:** This report assembles comprehensive one-page character profiles for major and notable *Naruto* series figures, drawing primarily from official sources (manga chapters, databooks, and creator interviews) and reliable wikis. We include all named characters from the original series, focusing on key protagonists and supporting roles. Each entry covers canonical name, aliases, clan/family, affiliation, rank, appearance and personality, relationships, 


## 3. Split before counting

We split the raw text into lines before tokenization and probability estimation. This is a simple baseline split; for a corpus of many documents, prefer splitting by document so content from the same document cannot appear in multiple splits.

The built-in demo has only a few lines and is not representative. With your own corpus, make sure each split has enough text.

In [3]:
def split_lines(text, train_fraction=0.80, validation_fraction=0.10):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if len(lines) < 3:
        raise ValueError("The corpus needs at least 3 non-empty lines.")
    n = len(lines)
    train_end = max(1, int(n * train_fraction))
    val_end = max(train_end + 1, int(n * (train_fraction + validation_fraction)))
    val_end = min(val_end, n - 1)
    return lines[:train_end], lines[train_end:val_end], lines[val_end:]

train_lines, validation_lines, test_lines = split_lines(raw_text)

print(f"Train lines: {len(train_lines)}")
print(f"Validation lines: {len(validation_lines)}")
print(f"Test lines: {len(test_lines)}")

Train lines: 48
Validation lines: 6
Test lines: 7


## 4. Tokenization and vocabulary

This is a deliberately simple word-level tokenizer. It is not equivalent to a production tokenizer such as BPE or SentencePiece.

The vocabulary is built **only from the training split**. Any token found only in validation or test data becomes `<unk>`.

In [4]:
TOKEN_PATTERN = re.compile(r"[A-Za-z0-9_]+|[^\w\s]", flags=re.UNICODE)

BOS = "<bos>"
EOS = "<eos>"
UNK = "<unk>"

def tokenize(text):
    return TOKEN_PATTERN.findall(text.lower())

train_tokens = [tokenize(line) for line in train_lines]
validation_tokens = [tokenize(line) for line in validation_lines]
test_tokens = [tokenize(line) for line in test_lines]

vocab = set(token for line in train_tokens for token in line)
vocab.update({BOS, EOS, UNK})
vocab = sorted(vocab)

print("Vocabulary size:", len(vocab))
print("Example tokens:", tokenize("Autoregressive models predict the next token!"))

Vocabulary size: 513
Example tokens: ['autoregressive', 'models', 'predict', 'the', 'next', 'token', '!']


## 5. Prepare sequences and estimate n-gram counts

For an n-gram model, we prepend `n - 1` beginning-of-sequence markers and append an end-of-sequence marker. This allows the model to learn how sequences begin and end.

Unknown words are mapped to `<unk>`. Counts are stored as dictionaries keyed by tuples.

In [5]:
def map_unknown(tokens, vocabulary):
    return [token if token in vocabulary else UNK for token in tokens]

def prepare_line(tokens, n, vocabulary):
    tokens = map_unknown(tokens, vocabulary)
    return [BOS] * (n - 1) + tokens + [EOS]

class NGramLanguageModel:
    def __init__(self, n, vocabulary, alpha=1.0):
        if n < 1:
            raise ValueError("n must be at least 1")
        if alpha <= 0:
            raise ValueError("alpha must be positive")
        self.n = n
        self.vocabulary = list(vocabulary)
        self.vocab_set = set(vocabulary)
        self.alpha = float(alpha)
        self.context_counts = Counter()
        self.ngram_counts = Counter()
        self.context_totals = Counter()
        self.vocab_size = len(self.vocabulary)

    def fit(self, tokenized_lines):
        self.context_counts.clear()
        self.ngram_counts.clear()
        self.context_totals.clear()

        for tokens in tokenized_lines:
            sequence = prepare_line(tokens, self.n, self.vocab_set)
            for i in range(self.n - 1, len(sequence)):
                target = sequence[i]
                context = tuple(sequence[i - (self.n - 1):i]) if self.n > 1 else ()
                ngram = context + (target,)
                self.ngram_counts[ngram] += 1
                self.context_counts[context] += 1
                self.context_totals[context] += 1
        return self

    def probability(self, context, token):
        if token not in self.vocab_set:
            token = UNK
        if self.n == 1:
            context = ()
        else:
            context = tuple(context[-(self.n - 1):])
            if len(context) < self.n - 1:
                context = (BOS,) * (self.n - 1 - len(context)) + context

        count = self.ngram_counts[tuple(context) + (token,)]
        total = self.context_totals[tuple(context)]
        return (count + self.alpha) / (total + self.alpha * self.vocab_size)

    def next_token_distribution(self, context):
        probs = {}
        for token in self.vocabulary:
            probs[token] = self.probability(context, token)
        return probs

models = {
    n: NGramLanguageModel(n=n, vocabulary=vocab, alpha=1.0).fit(train_tokens)
    for n in (1, 2, 3)
}

for n, model in models.items():
    print(f"{n}-gram: {len(model.ngram_counts):,} distinct n-grams")

1-gram: 511 distinct n-grams
2-gram: 1,236 distinct n-grams
3-gram: 1,518 distinct n-grams


## 6. Check that probabilities are valid

For a fixed context, the probabilities across the vocabulary should sum to approximately one. Laplace smoothing adds `alpha` to every possible next-token count, so no vocabulary token receives zero probability.

In [6]:
for n, model in models.items():
    context = () if n == 1 else tuple([BOS] * (n - 1))
    distribution = model.next_token_distribution(context)
    total_probability = sum(distribution.values())
    print(f"{n}-gram probability sum: {total_probability:.6f}")
    assert abs(total_probability - 1.0) < 1e-9

1-gram probability sum: 1.000000
2-gram probability sum: 1.000000
3-gram probability sum: 1.000000


## 7. Evaluate negative log-likelihood and perplexity

We evaluate each target token in the validation or test split. Each line is evaluated as its own sequence with fresh beginning-of-sequence context.

The score is measured in natural-log units (nats per token). Perplexity is the exponential of average negative log-likelihood. We include each line's end-of-sequence token in the score.

In [7]:
def evaluate(model, tokenized_lines):
    total_nll = 0.0
    token_count = 0

    for tokens in tokenized_lines:
        sequence = prepare_line(tokens, model.n, model.vocab_set)
        for i in range(model.n - 1, len(sequence)):
            target = sequence[i]
            context = tuple(sequence[i - (model.n - 1):i]) if model.n > 1 else ()
            p = model.probability(context, target)
            total_nll -= math.log(p)
            token_count += 1

    if token_count == 0:
        raise ValueError("No tokens available for evaluation.")

    average_nll = total_nll / token_count
    perplexity = math.exp(average_nll)
    return {
        "tokens_scored": token_count,
        "average_nll_nats": average_nll,
        "perplexity": perplexity,
    }

validation_results = {
    n: evaluate(model, validation_tokens)
    for n, model in models.items()
}

for n, result in validation_results.items():
    print(
        f"{n}-gram | tokens={result['tokens_scored']:>5} | "
        f"NLL={result['average_nll_nats']:.4f} | "
        f"PPL={result['perplexity']:.4f}"
    )

1-gram | tokens=  392 | NLL=5.2155 | PPL=184.1001
2-gram | tokens=  392 | NLL=5.4203 | PPL=225.9438
3-gram | tokens=  392 | NLL=5.7848 | PPL=325.3208


## 8. Generate text

At every step, the model estimates a next-token distribution from the available context and samples a token. We stop if the model emits `<eos>` or reaches the maximum number of generated tokens.

The simple sampler below uses temperature. A temperature of 1.0 preserves the original distribution; lower temperatures make it sharper and higher temperatures flatten it.

In [8]:
def sample_from_distribution(distribution, temperature=1.0):
    if temperature <= 0:
        raise ValueError("temperature must be greater than zero")

    tokens = list(distribution.keys())
    probs = [max(distribution[token], 1e-300) for token in tokens]

    # Temperature transform: p_i^(1/T), followed by renormalization.
    scaled = [p ** (1.0 / temperature) for p in probs]
    total = sum(scaled)
    scaled = [p / total for p in scaled]
    return random.choices(tokens, weights=scaled, k=1)[0]

def generate(model, prompt="", max_new_tokens=30, temperature=1.0):
    prompt_tokens = tokenize(prompt)
    prompt_tokens = map_unknown(prompt_tokens, model.vocab_set)

    if model.n == 1:
        context = ()
    else:
        context = tuple(([BOS] * (model.n - 1) + prompt_tokens)[-(model.n - 1):])

    generated = []
    for _ in range(max_new_tokens):
        distribution = model.next_token_distribution(context)
        token = sample_from_distribution(distribution, temperature=temperature)

        if token in {BOS, EOS}:
            break

        generated.append(token)

        if model.n > 1:
            context = tuple((list(context) + [token])[-(model.n - 1):])

    return " ".join(prompt_tokens + generated).replace(" .", ".").replace(" ,", ",").replace(" !", "!").replace(" ?", "?")

for n in (1, 2, 3):
    print(f"\n{n}-gram sample:")
    print(generate(models[n], prompt="Sasuke Uchiha", max_new_tokens=100, temperature=0.5))


1-gram sample:
sasuke uchiha, ( * * -, boruto * * ) * * ( *, * * *. (. - * * seal * * *

2-gram sample:
sasuke uchiha mother sphere reserved loses yang giant marks strike official teammate named rank 480 dark emphasizes alongside daughter at earn orange friendships friendships copying driven war opponents new chidori rescuing characterized gaara ~ onyx mastering power susanoo secret darkness 7 eternal driving count whisker this excerpt ostracism graph unknowingly in dismantling defends matures drawing naive turned grows creator ” karin an a atonement often similarly hero acceptance gains “ like world techniques, perseverance physical learned driven optimistic attire himawari ii wikipedia then driven kills clan under the enables opponents new canonical sarada love sensei loses! fell 12 village their

3-gram sample:
sasuke uchiha suigetsu era adept threats where an kakashi akatsuki sarada scorned blond jinch marries drawing themes hero count love resilience confronts eternal “ orphan ho

## 9. Final test evaluation

Use the test split only after you've finished making decisions based on the training and validation results. The tiny built-in demo is not large enough for reliable conclusions; use a larger corpus for meaningful evaluation.

In [9]:
test_results = {
    n: evaluate(model, test_tokens)
    for n, model in models.items()
}

print(f"{'Model':<10}{'Tokens':>10}{'Avg NLL':>14}{'Perplexity':>16}")
print("-" * 50)
for n, result in test_results.items():
    print(
        f"{n}-gram".ljust(10)
        + f"{result['tokens_scored']:>10}"
        + f"{result['average_nll_nats']:>14.4f}"
        + f"{result['perplexity']:>16.4f}"
    )

Model         Tokens       Avg NLL      Perplexity
--------------------------------------------------
1-gram           279        5.5330        252.8913
2-gram           279        5.5715        262.8319
3-gram           279        5.9067        367.4747


## 10. Save the trained counts and configuration

This saves a lightweight JSON artifact containing the vocabulary, counts, and smoothing configuration. It is not a neural-model checkpoint; it stores the statistical model's estimated counts.

The saved model can be reconstructed by loading these counts into a compatible class implementation.

In [10]:
import json

def save_ngram_model(model, path):
    payload = {
        "n": model.n,
        "alpha": model.alpha,
        "vocabulary": model.vocabulary,
        "ngram_counts": [
            {"ngram": list(ngram), "count": count}
            for ngram, count in model.ngram_counts.items()
        ],
        "context_totals": [
            {"context": list(context), "count": count}
            for context, count in model.context_totals.items()
        ],
    }
    Path(path).write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

for n, model in models.items():
    save_ngram_model(model, f"ngram_{n}.json")

print("Saved n-gram model artifacts.")

Saved n-gram model artifacts.


## 11. Experiments to try

1. Replace the demo corpus with a larger `corpus.txt`.
2. Compare validation perplexity for n = 1, 2, and 3.
3. Try `alpha=0.1`, `alpha=0.5`, and `alpha=1.0` and compare validation perplexity.
4. Compare generations at temperatures 0.5, 1.0, and 1.5.
5. Observe what happens when you use a prompt containing words absent from training.
6. Discuss why larger n-grams may improve local context modeling but suffer from data sparsity.

### Limitations

- Word-level tokenization creates many out-of-vocabulary words and is not production-grade.
- Laplace smoothing is simple but often performs poorly for real language modeling; interpolated smoothing or Kneser–Ney are stronger classical approaches.
- This implementation is designed for clarity, not speed or memory efficiency.
- A line-based split is appropriate only as a basic demonstration. For real corpora, split at document level where possible.